In [ ]:
!pip install numpy==1.26.4

In [ ]:
!pip install evaluate

In [ ]:
!pip install --upgrade transformers

In [ ]:
!pip install better_profanity

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Finetune RoBERTa for multi-label classification
import pandas as pd
import numpy as np
from datasets import Dataset
from pathlib import Path
import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

# Load Jigsaw dataset
df = pd.read_csv('./train.csv')

# Define binary toxicity label (toxic if any of the categories are 1)
df['toxic_label'] = (df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].sum(axis=1) > 0).astype(int)
df = df[['comment_text', 'toxic_label']]

# Split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenizer & model
checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(example):
    return tokenizer(example['comment_text'], truncation=True, padding='max_length', max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Set format for Trainer
train_dataset = train_dataset.rename_column("toxic_label", "label")
test_dataset = test_dataset.rename_column("toxic_label", "label")
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Evaluation metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

# Training configuration
training_args = TrainingArguments(
    output_dir="./roberta-toxic-jigsaw",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs",
    report_to=[]
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

# Save model
model.save_pretrained("roberta-toxic-jigsaw")
tokenizer.save_pretrained("roberta-toxic-jigsaw")

# Evaluate
results = trainer.evaluate()
print("Evaluation:", results)



In [ ]:
# Test
from better_profanity import profanity
profanity.load_censor_words()

model_dir = Path("/content/drive/My Drive/DALI/model")
# Load manually labeled dataset
lyrics_path = Path('/content/drive/MyDrive/DALI/clean_explicit_comparison/lyrics_labeled.csv')
lyrics_df = pd.read_csv(lyrics_path).dropna(subset=['text'])
feature_df = lyrics_df['text']
true_pred = lyrics_df['label']
texts = feature_df.tolist()

model_toxic_path = os.path.join("/content/drive/My Drive/DALI/model", 'roberta-toxic-jigsaw')
tokenizer_toxic = AutoTokenizer.from_pretrained(model_toxic_path)
model_toxic = AutoModelForSequenceClassification.from_pretrained(model_toxic_path)
model_toxic.eval()
inputs = tokenizer_toxic(texts, return_tensors="pt", truncation=True, padding=True, max_length=128)
with torch.no_grad():
    outputs = model_toxic(**inputs)
    preds = torch.argmax(outputs.logits, dim=1)
toxic_pred = preds.numpy()

In [ ]:
# Compare the results
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score
def print_results(true, pred):
  print("Precision: ", precision_score(true, pred))
  print("Recall: ", recall_score(true, pred))
  print("Accuracy: ", accuracy_score(true, pred))
  print("F1: ", f1_score(true, pred))

In [ ]:
# Only searching bad words
search_pred = lyrics_df['text'].astype(str).apply(profanity.contains_profanity)
search_pred = search_pred.to_numpy().astype(int)
print_results(true_pred,search_pred)

In [ ]:
# With finetuned language model
search_toxic_pred = np.logical_or(toxic_pred,search_pred).astype(int)
print_results(true_pred,search_toxic_pred)